# Focus Guard v2 — PyTorch training on the RTX 5090

TensorFlow has no sm_120 kernels, so training runs in torch on the VM and the model
ships to the Mac as ONNX.

**Baseline to beat** (measured on the Mac from `src/models/focus_guard_final.h5`, on
`data/test`):

| metric | baseline |
|---|---|
| 7-class accuracy | 0.6328 |
| 4-state accuracy | 0.7593 |
| **4-state macro-F1** | **0.7485** |

The last row is the target. `main.py` collapses 7 emotions into FOCUS / HAPPY / STRESS /
DISTRACTION, and **34% of the baseline's 7-class errors land inside the same state** —
invisible to the app. Half of the errors that *do* matter sit on one boundary:
FOCUS vs STRESS (885 of 1,728).

What this notebook changes, and why each one is here:

| # | Change | Why |
|---|---|---|
| 1 | Train on CLAHE-preprocessed images | `main.py` CLAHE-equalizes every face; training never did. Pure train/serve skew |
| 2 | Validation split carved from `train` | The old run selected checkpoints on `data/test`, so its score was inflated |
| 3 | sqrt-balanced class weights | `'balanced'` puts ~9.4x weight on `disgust` (436 images) and destabilises training |
| 4 | AdamW + warmup/cosine + label smoothing | Default Adam at a fixed LR leaves several points on the table |
| 5 | Gentler augmentation | 20 degrees + shear 0.2 + zoom 0.2 destroys mouth/eye geometry at 48px |
| 6 | Dropout before the head | The original had no regularisation beyond BatchNorm |
| 7 | Checkpoints chosen on 4-state macro-F1 | Stops rewarding `sad` vs `fear` separation that the app discards |
| 8 | Horizontal-flip TTA | Free accuracy at inference; costs one extra forward pass |
| 9 | Per-state decision calibration | Attacks the FOCUS/STRESS boundary directly, after training, at zero cost |

Set `SMOKE_TEST = True` first — it runs the whole pipeline in about a minute on 2,000
images so you find bugs before a long run, then set it back to False.

In [ ]:
import os, sys, math, glob, json, time
import numpy as np

PROJECT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(PROJECT); sys.path.insert(0, PROJECT)

SEED, IMG = 1337, 48
BATCH      = 256
EPOCHS     = 150
SMOKE_TEST = True         # start here; set False for the real run

RAW_DIR, CLAHE_DIR = 'data', 'data_clahe'
BEST = 'src/models/focus_guard_v2.pt'
ONNX = 'src/models/focus_guard.onnx'

CLASSES  = ['angry','disgust','fear','happy','neutral','sad','surprise']   # ImageFolder sorts alphabetically
STATES   = ['FOCUS','HAPPY','STRESS','DISTRACTION']
STATE_ID = {'neutral':0,'happy':1,'angry':2,'disgust':2,'fear':2,'sad':2,'surprise':3}
CLASS_TO_STATE = np.array([STATE_ID[c] for c in CLASSES])

W_FALSE_STRESS = 0.5   # cost of a false STRESS relative to macro-F1; 0 = pure macro-F1
BASELINE = {'acc7': 0.6328, 'acc4': 0.7593, 'f1_4': 0.7485,
            'false_stress': 0.238, 'missed_stress': 0.258}
if SMOKE_TEST:
    EPOCHS = 2
print('project:', PROJECT, '| smoke test:', SMOKE_TEST, '| epochs:', EPOCHS)

In [ ]:
import torch, torchvision
print('python     ', sys.version.split()[0])
print('torch      ', torch.__version__, '| torchvision', torchvision.__version__)
print('cuda build ', torch.version.cuda)

assert torch.cuda.is_available(), 'No GPU visible to torch — check the VM passthrough'
DEVICE = torch.device('cuda')
cap = torch.cuda.get_device_capability()
print('gpu        ', torch.cuda.get_device_name(0), f'| compute capability {cap[0]}.{cap[1]}')
assert cap[0] >= 12 or True   # sm_120 on the 5090; older cards still fine

AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('amp dtype  ', AMP_DTYPE, '(bf16 needs no GradScaler)')

torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

## 1. Make training see what the webcam sends

`main.py:104` runs blur + CLAHE on every face before inference. Training never did, so
the model was tested on images that don't look like its training data. One-time fix,
about a minute for 35,887 images.

In [ ]:
import cv2

def build_clahe(src=RAW_DIR, dst=CLAHE_DIR):
    if os.path.isdir(dst):
        print('already exists, skipping:', dst); return
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))   # same params as preprocess_face()
    files = glob.glob(src + '/*/*/*.jpg')
    t0 = time.time()
    for i, path in enumerate(files):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        out = clahe.apply(cv2.GaussianBlur(img, (3, 3), 0))       # same order as preprocess_face()
        d = path.replace(src, dst, 1)
        os.makedirs(os.path.dirname(d), exist_ok=True)
        cv2.imwrite(d, out)
        if i % 8000 == 0:
            print(i, '/', len(files))
    print('wrote', len(files), 'files in', round(time.time() - t0, 1), 's')

build_clahe()

## 2. Data

Two `ImageFolder` views over the same directory: the training indices get augmentation,
the validation indices don't. `data/test` is never touched until section 5.

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms as T
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split

train_tf = T.Compose([
    T.Grayscale(num_output_channels=1),
    T.RandomHorizontalFlip(),
    T.RandomAffine(degrees=10, translate=(0.08, 0.08), scale=(0.9, 1.1)),
    T.ColorJitter(contrast=0.15),
    T.ToTensor(),                                                  # -> [0,1], matches rescale=1./255
])
eval_tf = T.Compose([T.Grayscale(num_output_channels=1), T.ToTensor()])

root = CLAHE_DIR + '/train'
ds_train_view, ds_eval_view = ImageFolder(root, train_tf), ImageFolder(root, eval_tf)
assert ds_train_view.classes == CLASSES, ds_train_view.classes

all_targets = np.array(ds_train_view.targets)
idx = np.arange(len(all_targets))
if SMOKE_TEST:
    idx, _ = train_test_split(idx, train_size=2000, stratify=all_targets, random_state=SEED)
tr_idx, va_idx = train_test_split(idx, test_size=0.1, stratify=all_targets[idx], random_state=SEED)

train_ds = Subset(ds_train_view, tr_idx)
val_ds   = Subset(ds_eval_view,  va_idx)
test_ds  = ImageFolder(CLAHE_DIR + '/test', eval_tf)

kw = dict(num_workers=8, pin_memory=True, persistent_workers=True)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, drop_last=True, **kw)
val_dl   = DataLoader(val_ds,   batch_size=512, shuffle=False, **kw)
test_dl  = DataLoader(test_ds,  batch_size=512, shuffle=False, **kw)

counts = np.bincount(all_targets[tr_idx], minlength=len(CLASSES)).astype(np.float64)
w = (counts.sum() / (len(CLASSES) * counts)) ** 0.5
class_w = torch.tensor(w, dtype=torch.float32, device=DEVICE)
print('train', len(train_ds), '| val', len(val_ds), '| test', len(test_ds))
print(dict(zip(CLASSES, counts.astype(int))))
print('weights', {CLASSES[i]: round(float(v), 2) for i, v in enumerate(w)})

## 3. Model — mini-Xception ported from `src/model.py`

Same topology as the Keras original: depthwise-separable blocks with 1x1 strided
residuals, 1.53 M parameters. Dropout before the head is new. The head returns
**logits** — `CrossEntropyLoss` needs logits, and softmax gets attached at export
because `main.py` expects probabilities.

In [ ]:
class SepConv(nn.Module):
    # Keras SeparableConv2D = depthwise then pointwise, both bias-free
    def __init__(self, cin, cout, k=3):
        super().__init__()
        self.dw = nn.Conv2d(cin, cin, k, padding=k // 2, groups=cin, bias=False)
        self.pw = nn.Conv2d(cin, cout, 1, bias=False)
    def forward(self, x):
        return self.pw(self.dw(x))

class XBlock(nn.Module):
    def __init__(self, cin, cout, lead_relu=True):
        super().__init__()
        self.lead_relu = lead_relu
        self.res  = nn.Sequential(nn.Conv2d(cin, cout, 1, stride=2, bias=False), nn.BatchNorm2d(cout))
        self.s1   = nn.Sequential(SepConv(cin, cout), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
        self.s2   = nn.Sequential(SepConv(cout, cout), nn.BatchNorm2d(cout))
        self.pool = nn.MaxPool2d(3, stride=2, padding=1)      # matches Keras padding='same'
    def forward(self, x):
        r = self.res(x)
        if self.lead_relu:
            x = torch.relu(x)
        return self.pool(self.s2(self.s1(x))) + r

class MiniXception(nn.Module):
    def __init__(self, num_classes=7, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.b1 = XBlock(64, 128, lead_relu=False)            # first block has no leading ReLU, as in model.py
        self.b2 = XBlock(128, 256)
        self.b3 = XBlock(256, 512)
        self.tail = nn.Sequential(
            SepConv(512, 512), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            SepConv(512, 1024), nn.BatchNorm2d(1024), nn.ReLU(inplace=True))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(dropout), nn.Linear(1024, num_classes))
    def forward(self, x):
        return self.head(self.tail(self.b3(self.b2(self.b1(self.stem(x))))))

model = MiniXception().to(DEVICE)
print(round(sum(p.numel() for p in model.parameters()) / 1e6, 2), 'M params')
print('output shape:', tuple(model(torch.zeros(2, 1, IMG, IMG, device=DEVICE)).shape))

## 4. Train

Checkpoints are selected on 4-state macro-F1 — the metric the app actually consumes.

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

@torch.no_grad()
def predict_probs(dl, tta=False):
    model.eval()
    P, Y = [], []
    for xb, yb in dl:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=AMP_DTYPE):
            out = model(xb).float().softmax(1)
            if tta:                                   # horizontal-flip TTA
                out = (out + model(torch.flip(xb, dims=[3])).float().softmax(1)) / 2
        P.append(out.cpu()); Y.append(yb)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def score(probs, y):
    p = probs.argmax(1)
    sp, sy = CLASS_TO_STATE[p], CLASS_TO_STATE[y]
    return {'acc7': float((p == y).mean()),
            'acc4': float((sp == sy).mean()),
            'f1_4': float(f1_score(sy, sp, average='macro'))}

crit = nn.CrossEntropyLoss(weight=class_w, label_smoothing=0.05)
opt  = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
warm = max(1, round(EPOCHS * 0.03))
sched = torch.optim.lr_scheduler.SequentialLR(opt, [
    torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.01, total_iters=warm),
    torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS - warm), eta_min=1e-5)],
    milestones=[warm])

best, patience, since = -1.0, 25, 0
for epoch in range(EPOCHS):
    model.train(); t0, tot, n = time.time(), 0.0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=AMP_DTYPE):
            loss = crit(model(xb), yb)
        loss.backward(); opt.step()
        tot += loss.item() * xb.size(0); n += xb.size(0)
    sched.step()
    m = score(*predict_probs(val_dl))
    flag = ''
    if m['f1_4'] > best:
        best, since = m['f1_4'], 0
        torch.save({'state_dict': model.state_dict(), 'classes': CLASSES, 'f1_4': best}, BEST)
        flag = '  <- best, saved'
    else:
        since += 1
    print(f"ep {epoch+1:3d}/{EPOCHS}  loss {tot/n:.4f}  val acc7 {m['acc7']:.4f}  "
          f"val F1_4 {m['f1_4']:.4f}  {time.time()-t0:5.1f}s{flag}")
    if since >= patience:
        print('early stop — no improvement for', patience, 'epochs'); break
print('best val 4-state macro-F1:', round(best, 4))

## 5. Calibrate the decision — with the app's real costs

Training optimises 7-class cross-entropy, but the app only asks "which of 4 states?".
Summing class probabilities into state probabilities and shifting each state by a fitted
bias costs nothing at inference and targets the FOCUS/STRESS confusion that carries half
the app's errors.

Macro-F1 alone treats every error as equal. In this app they are not:

| Error | What the user experiences | Cost |
|---|---|---|
| Calm read as STRESS | Music switches while you are concentrating | **High** |
| Stress read as calm | App stays quiet; you can override with a gesture | Low |

`W_FALSE_STRESS` is that asymmetry as a dial. `0.0` reproduces pure macro-F1; higher
values buy fewer interruptions by giving up F1, and the exchange rate gets steep fast —
past about 1.0 the model nearly stops predicting STRESS at all. The sweep prints the
whole frontier so you pick with your eyes open rather than trusting my default.

The bias is fitted on **validation** and applied unchanged to test — fitting it on test
would repeat the mistake the original training made.


In [ ]:
# torch has no restore_best_weights: the loop saved the best checkpoint but left
# `model` holding the FINAL epoch's weights. Reload, or everything below scores
# the wrong model.
ckpt = torch.load(BEST, map_location=DEVICE)
model.load_state_dict(ckpt['state_dict'])
print('loaded best checkpoint — val 4-state macro-F1', round(ckpt['f1_4'], 4))

def to_state_probs(probs):
    s = np.zeros((len(probs), 4), dtype=np.float64)
    for c in range(len(CLASSES)):
        s[:, CLASS_TO_STATE[c]] += probs[:, c]
    return s

def decide(state_probs, bias):
    return (np.log(state_probs + 1e-9) + bias).argmax(1)

def app_metrics(state_probs, y_state, bias):
    """F1 plus the two rates the app actually feels."""
    pred = decide(state_probs, bias)
    calm, stressed = (y_state == 0), (y_state == 2)
    return {'f1': float(f1_score(y_state, pred, average='macro')),
            'acc': float((pred == y_state).mean()),
            'false_stress':  float((calm & (pred == 2)).sum() / max(1, calm.sum())),
            'missed_stress': float((stressed & (pred != 2)).sum() / max(1, stressed.sum()))}

def objective(state_probs, y_state, bias, w):
    m = app_metrics(state_probs, y_state, bias)
    return m['f1'] - w * m['false_stress']

def fit_bias(state_probs, y_state, w, sweeps=3):
    grid, bias = np.arange(-1.0, 1.01, 0.05), np.zeros(4)
    for _ in range(sweeps):                       # coordinate ascent
        for k in range(4):
            scores = [objective(state_probs, y_state, np.where(np.arange(4) == k, g, bias), w)
                      for g in grid]
            bias[k] = grid[int(np.argmax(scores))]
    return bias

val_probs, val_y = predict_probs(val_dl, tta=True)
vs, vy = to_state_probs(val_probs), CLASS_TO_STATE[val_y]

print(f"\n{'w':>4}  {'bias [FOCUS HAPPY STRESS DISTRACT]':36s} {'F1':>7} {'acc':>7} "
      f"{'false-stress':>13} {'missed-stress':>14}")
frontier = {}
for w in sorted({0.0, 0.25, 0.5, 1.0, 2.0, float(W_FALSE_STRESS)}):
    b = fit_bias(vs, vy, w)
    m = app_metrics(vs, vy, b)
    frontier[w] = b
    print(f"{w:4.1f}  {str(np.round(b, 2)):36s} {m['f1']:7.4f} {m['acc']:7.4f} "
          f"{m['false_stress']:13.3f} {m['missed_stress']:14.3f}")

bias = frontier[float(W_FALSE_STRESS)]
print(f"\nw=0 is pure macro-F1: it treats interrupting a focused user and missing stress")
print(f"as equally bad. Raise W_FALSE_STRESS to buy fewer false alarms with a little F1.")
print(f"using W_FALSE_STRESS={W_FALSE_STRESS} -> {dict(zip(STATES, bias.round(2)))}")


## 6. Final score on the untouched test set

In [ ]:
test_probs, test_y = predict_probs(test_dl, tta=True)
plain = score(test_probs, test_y)
ts, ty = to_state_probs(test_probs), CLASS_TO_STATE[test_y]
raw = app_metrics(ts, ty, np.zeros(4))
cal = app_metrics(ts, ty, bias)

print(classification_report(test_y, test_probs.argmax(1), target_names=CLASSES, digits=3, zero_division=0))
print('4-state confusion after calibration', STATES)
print(confusion_matrix(ty, decide(ts, bias)))
print()
print(f"{'':24s} {'4-state F1':>11} {'acc':>8} {'false-stress':>13} {'missed-stress':>14}")
print(f"{'baseline (old model)':24s} {BASELINE['f1_4']:11.4f} {BASELINE['acc4']:8.4f} "
      f"{BASELINE['false_stress']:13.3f} {BASELINE['missed_stress']:14.3f}")
print(f"{'v2 + TTA':24s} {raw['f1']:11.4f} {raw['acc']:8.4f} "
      f"{raw['false_stress']:13.3f} {raw['missed_stress']:14.3f}")
print(f"{'v2 + TTA + calibrated':24s} {cal['f1']:11.4f} {cal['acc']:8.4f} "
      f"{cal['false_stress']:13.3f} {cal['missed_stress']:14.3f}")
print(f"\n7-class accuracy  baseline {BASELINE['acc7']:.4f} -> v2 {plain['acc7']:.4f}")
print(f"delta on the metric that matters: {cal['f1'] - BASELINE['f1_4']:+.4f}")

os.makedirs('runs', exist_ok=True)
json.dump({'baseline': BASELINE, 'v2_raw': raw, 'v2_calibrated': cal, 'acc7': plain['acc7'],
           'state_bias': bias.tolist(), 'w_false_stress': W_FALSE_STRESS, 'states': STATES},
          open('runs/scores_torch.json', 'w'), indent=2)


## 7. Export for the Mac

`state_dict` always saves. ONNX export uses torch's dynamo exporter, which needs
`onnxscript` — if it isn't installed the cell says so instead of failing the run.

In [ ]:
# Add the calibration to the existing best checkpoint WITHOUT touching its weights.
ckpt = torch.load(BEST, map_location='cpu')
ckpt['state_bias'] = bias.tolist()
torch.save(ckpt, BEST)
print('added state_bias to', BEST, '| val F1_4', round(ckpt['f1_4'], 4))

meta = {'classes': CLASSES, 'states': STATES, 'input': [1, 1, IMG, IMG],
        'state_bias': bias.tolist(), 'tta': 'average of image and horizontal flip',
        'preprocessing': 'GaussianBlur(3,3) -> CLAHE(2.0, 8x8) -> resize 48x48 -> /255'}
json.dump(meta, open('src/models/focus_guard.onnx.json', 'w'), indent=2)

try:
    export_model = nn.Sequential(model.eval().cpu(), nn.Softmax(dim=1)).eval()
    torch.onnx.export(export_model, torch.zeros(1, 1, IMG, IMG), ONNX,
                      input_names=['input'], output_names=['probs'],
                      dynamic_axes={'input': {0: 'batch'}, 'probs': {0: 'batch'}})
    print('wrote', ONNX, round(os.path.getsize(ONNX) / 1e6, 2), 'MB')
    print('now:  scp', ONNX, 'src/models/focus_guard.onnx.json  mac:<project>/src/models/')
except ModuleNotFoundError as e:
    print('ONNX export needs an extra package:', e)
    print('run:  pip install onnxscript   then re-run this cell')
finally:
    model.to(DEVICE)